In [ ]:
#!/usr/bin/env python

import os
import re
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from datetime import datetime
from isoweek import Week
from cmdstanpy import CmdStanModel
from cmdstanpy import from_csv
import glob

from numba import njit
from scipy.signal import savgol_filter

In [ ]:
@njit
def get_noise_draws(noise_level, T, dt):
    alpha = dt / noise_level**2
    beta = noise_level**2
    k_shape = alpha
    theta_scale = beta
    noise_draws = np.empty(T)
    for i in range(T):
        DeltaGamma_loc = np.random.gamma(k_shape, theta_scale)
        noise_draws[i] = DeltaGamma_loc / dt
    return noise_draws

@njit
def sirs_ode_stochastic(t, S, I, R, beta0, dbeta, betaphase, gamma, mu, delta, noise_draw):
    beta_t = beta0 * (1 + dbeta * np.sin(2 * np.pi * t / 52 + betaphase)) * noise_draw
    dSdt = mu - beta_t * S * I - mu * S + delta * R
    dIdt = beta_t * S * I - (mu + gamma) * I
    dRdt = gamma * I - (mu + delta) * R
    return dSdt, dIdt, dRdt, beta_t

In [ ]:
def sirs_trajectory(beta0, dbeta, betaphase, gamma, mu, delta,
                                      S0, I0, dt=0.05, total_time=20 * 52,
                                      noise_level=0.11):
    N = int(total_time / dt)
    S, I, R = S0, I0, 1.0 - S0 - I0
    
    
    noise_draws = get_noise_draws(noise_level, N, dt)
    Ss = np.zeros(N)
    Is = np.zeros(N)
    Rs = np.zeros(N)
    
    t = 0.0
    ts = []
    for k in range(N):
        dS, dI, dR, beta_t = sirs_ode_stochastic(t, S, I, R, beta0, dbeta, betaphase,
                                                 gamma, mu, delta, noise_draws[k])

        # Update base trajectory using Euler
        S += dt * dS
        I += dt * dI
        R += dt * dR
        
        Ss[k] = S
        Is[k] = I
        Rs[k] = R

        t += dt
        ts.append(t)

    return Ss, Is, Rs, np.array(ts)

In [ ]:
csv_files = glob.glob('stan_output/sinusoid_2025/*20250801092152*.csv')

print(csv_files)

fit = from_csv(csv_files)

In [ ]:
# These are just the individual means:
for var in ['S0', 'logx_I0', 'beta0', 'dbeta', 'betaphase', 'sigma_obs', 'rho', 'delta']:
#for var in ['S0', 'I0', 'beta0', 'dbeta', 'betaphase', 'sigma_obs', 'rho', 'tau']:
    print(f"{var}: {fit.stan_variable(var).mean()}")

In [ ]:
df_draws = fit.draws_pd()
lp = df_draws["lp__"]  # log posterior column
imax = np.argmax(lp)  # index of sample with highest lp__
best_draw = df_draws.iloc[imax]

In [ ]:
best_draw

In [ ]:
def best_draw_to_dict(row):
    """
    Convert a series of Stan draws into a structured dict,
    grouping vector parameters like param[1], param[2], ... into arrays.
    """
    result = {}
    for col in row.index:
        # Attempt to match something like "paramName[123]"
        m = re.match(r"(.*)\[(\d+)\]$", col)
        if m:
            base_name = m.group(1)
            idx = int(m.group(2))  # 1-based index from Stan
            val = row[col]

            if base_name not in result:
                # store them in a dict first, convert to array after.
                result[base_name] = {}
            result[base_name][idx] = val
        else:
            # it is a scalar param or a special column like chain__, iter__, ...
            result[col] = row[col]

    # Convert any dict-of-indices to a list or array
    # Stan uses 1-indexing
    for k, v in list(result.items()):
        if isinstance(v, dict):
            # v is a dictionary of indices -> values
            max_idx = max(v.keys())
            # create an array of length = max index
            arr = np.empty(max_idx, dtype=float)
            for i in range(1, max_idx + 1):
                arr[i - 1] = v[i]  # shift to zero-indexing
            result[k] = arr
    return result

best_draw_dict = best_draw_to_dict(best_draw)

In [ ]:
def DiscTimeRate(r, dt):
    return (1 - np.exp(-r*dt))/dt

In [ ]:
dt_discrete = 1/8.0

mu = DiscTimeRate(1/(52 * 80.0), dt_discrete)

with_seasonality = True

beta0 = DiscTimeRate(best_draw_dict['beta0'], dt_discrete)
betaphase = best_draw_dict['betaphase']
gamma = DiscTimeRate(best_draw_dict['gamma'], dt_discrete)
delta = DiscTimeRate(best_draw_dict['delta'], dt_discrete)
S0 = best_draw_dict['S0']
I0 = best_draw_dict['I0']



if with_seasonality:
    dbeta = best_draw_dict['dbeta']
else:
    dbeta = 0.0



In [ ]:
n_weeks = 52*100

In [ ]:
S, I, R, ts = sirs_trajectory(
            beta0=beta0,  # This should already be DiscRateToCont
            dbeta=dbeta,
            betaphase=betaphase,
            gamma=gamma,
            mu=mu,
            delta=delta,
            S0=S0,
            I0=I0,
            dt=0.1,
            total_time=n_weeks,
            noise_level=0.12
        )

#I = savgol_filter(I, window_length=52*5*10, polyorder=3)

plt.figure(figsize=(4, 2.5), dpi=300)

plt.plot(ts/52, 100*I)
plt.ylabel("Prevalence (%)")
plt.xlabel("Time (years)")
plt.xlim([0,50])

In [ ]:
# Autocorrelation function
def autocorrelation(I, ABS=False):
    I = I - np.mean(I)
    N = len(I)
    result = np.correlate(I, I, mode='full')[N-1:]
    normalization = np.arange(N, 0, -1)
    result = result / normalization
    result = result / result[0]  # normalize to 1 at lag 0
    if ABS:
        result = np.abs(result)
    return result

In [ ]:
nth = 10

ac = autocorrelation(I[::nth], ABS=True)
lags = (ts - ts[0])[::nth]

plt.plot(lags/52, ac)

In [ ]:
nth = 40
lags = (ts - ts[0])[::nth]

n_repeats = 300

dt_sim = 0.1

for n in range(n_repeats):
    S, I, R, ts = sirs_trajectory(
            beta0=beta0,  # This should already be DiscRateToCont
            dbeta=dbeta,
            betaphase=betaphase,
            gamma=gamma,
            mu=mu,
            delta=delta,
            S0=S0,
            I0=I0,
            dt=dt_sim,
            total_time=n_weeks,
            noise_level=0.12
        )
    
    # Smoothing, so only the multiyear patterns persist
    I_smoo = savgol_filter(I, window_length=52*2*int(1.0/dt_sim), polyorder=3)
    
    if n==0:
        ac = autocorrelation(I[::nth])/n_repeats
        ac_smoo = autocorrelation(I_smoo[::nth])/n_repeats
    else:
        ac += autocorrelation(I[::nth])/n_repeats
        ac_smoo += autocorrelation(I_smoo[::nth])/n_repeats


In [ ]:
# Unsmoothed autocorrelation of unsmoothed signal

plt.figure(figsize=(4, 2.5), dpi=300)
ABS = True
lw=0.5
if ABS:
    plt.plot(lags/52, np.abs(ac), lw=lw)
else:
    plt.plot(lags/52, ac, lw=lw)
plt.xlim([0, 90])
plt.xlabel("Lag (years)")
if ABS:
    plt.ylabel(r"$\vert$Autocorrelation$\vert$")
else:
    plt.ylabel(r"Autocorrelation")
plt.ylim([-0.02,1.02])

In [ ]:
# Unsmoothed autocorrelation of smoothed signal

plt.figure(figsize=(4, 2.5), dpi=300)
ABS = True
if ABS:
    plt.plot(lags/52, np.abs(ac_smoo))
else:
    plt.plot(lags/52, ac_smoo)
plt.xlim([0, 90])
plt.xlabel("Lag (years)")
if ABS:
    plt.ylabel(r"$\vert$Autocorrelation$\vert$")
else:
    plt.ylabel(r"Autocorrelation")
plt.ylim([-0.02,1.02])

In [ ]:
# Smoothed autocorrelation of UNsmoothed signal

from scipy.signal import savgol_filter





plt.figure(figsize=(3,2.3), dpi=300)
# Apply Savitzky-Golay filter
if ABS:
    ac_smoothed = savgol_filter(np.abs(ac), window_length=52*2, polyorder=4)
else:
    ac_smoothed = savgol_filter(ac, window_length=52*2, polyorder=4)
ac_smoothed[0]=1
plt.plot(lags/52, ac_smoothed, label="AC (original signal)", color="purple")
plt.xlim([0, 50])
plt.ylim([0,1])
plt.ylabel(r"$|$Autocorrelation$|$" if ABS else "Autocorrelation")
plt.xlabel("Lag (years)")
plt.legend()
plt.tight_layout()

In [ ]:
# Smoothed autocorrelation of smoothed signal

from scipy.signal import savgol_filter
plt.figure(figsize=(4, 2.5), dpi=300)
# Apply Savitzky-Golay filter
if ABS:
    ac_smoothed = savgol_filter(np.abs(ac_smoo), window_length=52*2, polyorder=4)
else:
    ac_smoothed = savgol_filter(ac_smoo, window_length=52*2, polyorder=4)
ac_smoothed[0]=1
plt.plot(lags/52, ac_smoothed)
plt.xlim([0, 90])
plt.ylim([0,1])
if ABS:
    plt.ylabel(r"$\vert$Autocorrelation$\vert$")
else:
    plt.ylabel(r"Autocorrelation")

In [ ]:
from scipy.optimize import curve_fit

# Convert lags to years
lags_years = lags / 52

# Restrict to first n years
mask = (lags_years > 0) & (lags_years <= 50)
xdata = lags_years[mask]
ydata = ac_smoothed[mask]

# Exponential decay model: A * exp(-t/T) + C
def exp_decay(t, A, T, C):
    return A * np.exp(-t / T) + C

# Initial guesses: A≈1, T≈10 years, C≈0
p0 = [1.0, 10.0, 0.0]

# Fit
params, cov = curve_fit(exp_decay, xdata, ydata, p0=p0, bounds=([0,0,-1],[2,200,1]))
A_fit, T_fit, C_fit = params

print(f"Fit parameters:\n A = {A_fit:.3f}, T = {T_fit:.2f} years, C = {C_fit:.3f}")

# Plot
plt.figure(figsize=(3,2.3), dpi=300)
plt.plot(lags_years, ac_smoothed, label="AC (smoothed signal)", color="purple")
plt.plot(xdata, exp_decay(xdata, *params), label=r"Exp fit ($T_{1/2}=$"+f"{np.log(2)*T_fit:.1f} y)", color="orange")
plt.xlim([0, 50])
plt.ylim([0, 1])
plt.xlabel("Lag (years)")
plt.ylabel("Autocorrelation" if not ABS else r"$|$Autocorrelation$|$")
plt.legend()
plt.tight_layout()
plt.show()

